In [1]:
import pandas as pd

In [2]:
matches_df = pd.read_csv('data/raw/matches.csv')
deliveries_df = pd.read_csv('data/raw/deliveries.csv')

In [3]:
print("Matches Data Shape:", matches_df.shape)
display(matches_df.head())

Matches Data Shape: (1095, 20)


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [4]:
print("Deliveries Data Shape:", deliveries_df.shape)
display(deliveries_df.head())

Deliveries Data Shape: (260920, 17)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [5]:
print(matches_df.isnull().sum())

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64


In [6]:
print(matches_df['team1'].unique())

['Royal Challengers Bangalore' 'Kings XI Punjab' 'Delhi Daredevils'
 'Mumbai Indians' 'Kolkata Knight Riders' 'Rajasthan Royals'
 'Deccan Chargers' 'Chennai Super Kings' 'Kochi Tuskers Kerala'
 'Pune Warriors' 'Sunrisers Hyderabad' 'Gujarat Lions'
 'Rising Pune Supergiants' 'Rising Pune Supergiant' 'Delhi Capitals'
 'Punjab Kings' 'Lucknow Super Giants' 'Gujarat Titans'
 'Royal Challengers Bengaluru']


In [7]:
# 1. Define a dictionary to map old franchise names to their current active names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad' 
}

# Apply the mapping to all relevant columns in the matches dataset
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches_df[col] = matches_df[col].replace(team_mapping)

# Apply the mapping to the deliveries dataset
for col in ['batting_team', 'bowling_team']:
    deliveries_df[col] = deliveries_df[col].replace(team_mapping)

# 2. Define the exact list of the 10 currently active IPL franchises
active_teams = [
    'Chennai Super Kings',
    'Delhi Capitals',
    'Gujarat Titans',
    'Kolkata Knight Riders',
    'Lucknow Super Giants',
    'Mumbai Indians',
    'Punjab Kings',
    'Rajasthan Royals',
    'Royal Challengers Bengaluru',
    'Sunrisers Hyderabad'
]

# 3. Filter the matches dataframe to only include games between active teams
matches_df = matches_df[(matches_df['team1'].isin(active_teams)) & 
                        (matches_df['team2'].isin(active_teams))]

# 4. Filter the deliveries dataframe to only include action from active teams
deliveries_df = deliveries_df[(deliveries_df['batting_team'].isin(active_teams)) & 
                              (deliveries_df['bowling_team'].isin(active_teams))]

# Display the clean results
print("Cleaned Matches Shape:", matches_df.shape)
print("Active Teams in Dataset:", matches_df['team1'].unique())

Cleaned Matches Shape: (980, 20)
Active Teams in Dataset: ['Royal Challengers Bengaluru' 'Punjab Kings' 'Delhi Capitals'
 'Mumbai Indians' 'Kolkata Knight Riders' 'Rajasthan Royals'
 'Sunrisers Hyderabad' 'Chennai Super Kings' 'Lucknow Super Giants'
 'Gujarat Titans']


In [8]:
# 1. Remove matches where the Duckworth-Lewis method was applied
# In most Kaggle IPL datasets, this is indicated by a 'dl_applied' column (1 = yes, 0 = no)
if 'dl_applied' in matches_df.columns:
    matches_df = matches_df[matches_df['dl_applied'] == 0]

# 2. Rename the 'id' column in matches_df to 'match_id' to match deliveries_df
matches_df = matches_df.rename(columns={'id': 'match_id'})

# 3. Calculate the total runs scored in the 1st innings of every match
innings1_df = deliveries_df[deliveries_df['inning'] == 1]
total_score_df = innings1_df.groupby(['match_id'])['total_runs'].sum().reset_index()

# 4. Calculate the target score (1st innings total + 1)
total_score_df['target'] = total_score_df['total_runs'] + 1

# 5. Merge the target score back into the main matches dataframe
matches_df = matches_df.merge(total_score_df[['match_id', 'target']], on='match_id')

# Display the updated dataframe
print("Matches Shape after D/L removal:", matches_df.shape)
display(matches_df[['match_id', 'city', 'winner', 'target']].head())

Matches Shape after D/L removal: (980, 21)


,match_id,city,winner,target
0,335982,Bangalore,Kolkata Knight Riders,223
1,335983,Chandigarh,Chennai Super Kings,241
2,335984,Delhi,Delhi Capitals,130
3,335985,Mumbai,Royal Challengers Bengaluru,166
4,335986,Kolkata,Kolkata Knight Riders,111


In [9]:
import numpy as np

# 1. Isolate the second innings (the run chase)
chase_df = deliveries_df[deliveries_df['inning'] == 2].copy()

# 2. Merge essential match details (including target) into the chase dataframe
match_cols = ['match_id', 'city', 'winner', 'target']
chase_df = chase_df.merge(matches_df[match_cols], on='match_id')

# 3. Calculate Current Score (Cumulative sum of runs per match)
chase_df['current_score'] = chase_df.groupby('match_id')['total_runs'].cumsum()

# 4. Calculate Runs Left
chase_df['runs_left'] = chase_df['target'] - chase_df['current_score']
chase_df['runs_left'] = chase_df['runs_left'].apply(lambda x: 0 if x < 0 else x) # Handle negative runs left

# 5. Calculate Balls Bowled and Balls Left
# Most Kaggle IPL datasets index overs starting at 1. 
chase_df['balls_bowled'] = (chase_df['over'] - 1) * 6 + chase_df['ball']
chase_df['balls_left'] = 120 - chase_df['balls_bowled']
chase_df['balls_left'] = chase_df['balls_left'].apply(lambda x: 0 if x < 0 else x) # Handle extras going over 120

# 6. Calculate Wickets Remaining
# If 'player_dismissed' is not null, a wicket fell.
chase_df['is_wicket'] = chase_df['player_dismissed'].apply(lambda x: 1 if pd.notnull(x) else 0)
chase_df['cumulative_wickets'] = chase_df.groupby('match_id')['is_wicket'].cumsum()
chase_df['wickets_remaining'] = 10 - chase_df['cumulative_wickets']

# 7. Calculate Current Run Rate (CRR)
chase_df['crr'] = (chase_df['current_score'] * 6) / chase_df['balls_bowled']

# 8. Calculate Required Run Rate (RRR)
# Use np.where to prevent division by zero on the final ball
chase_df['rrr'] = np.where(chase_df['balls_left'] == 0, 0, (chase_df['runs_left'] * 6) / chase_df['balls_left'])

# 9. Define the Target Variable (1 for win, 0 for loss)
chase_df['result'] = chase_df.apply(lambda row: 1 if row['batting_team'] == row['winner'] else 0, axis=1)

# Display the final engineered features
columns_to_view = ['batting_team', 'bowling_team', 'current_score', 'runs_left', 'balls_left', 'wickets_remaining', 'crr', 'rrr', 'result']
display(chase_df[columns_to_view].tail())

,batting_team,bowling_team,current_score,runs_left,balls_left,wickets_remaining,crr,rrr,result
112800,Kolkata Knight Riders,Sunrisers Hyderabad,110,4,67,8,12.452830,0.358209,1
112801,Kolkata Knight Riders,Sunrisers Hyderabad,111,3,66,8,12.333333,0.272727,1
112802,Kolkata Knight Riders,Sunrisers Hyderabad,112,2,65,8,12.218182,0.184615,1
112803,Kolkata Knight Riders,Sunrisers Hyderabad,113,1,64,8,12.107143,0.093750,1
112804,Kolkata Knight Riders,Sunrisers Hyderabad,114,0,63,8,12.000000,0.000000,1


In [10]:
from sklearn.model_selection import train_test_split

# 1. Define the VIP list of features required for the ML model
final_columns = [
    'batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 
    'wickets_remaining', 'target', 'crr', 'rrr', 'result'
]

# 2. Extract only these columns to drop all the noise (umpires, extras, etc.)
final_df = chase_df[final_columns].copy()

# 3. Handle Missing Values
# The 'city' column is often null for UAE matches. We fill those, then drop any other bad rows.
final_df['city'] = final_df['city'].fillna('Dubai')
final_df = final_df.dropna()

# 4. Filter out unrealistic scenarios (e.g., 0 balls left which breaks the math)
final_df = final_df[final_df['balls_left'] > 0]

# 5. Shuffle the dataset
# This prevents the model from learning a chronological bias.
final_df = final_df.sample(final_df.shape[0], random_state=42)

# 6. Split into Features (X) and Target (y)
X = final_df.drop(columns=['result'])
y = final_df['result']

# 7. Perform the Train-Test Split
# 80% for training the model, 20% held back for testing its accuracy.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shapes of the final matrices
print("Training Features Shape (X_train):", X_train.shape)
print("Testing Features Shape (X_test):", X_test.shape)

Training Features Shape (X_train): (90222, 9)
Testing Features Shape (X_test): (22556, 9)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Define the Column Transformer
# 'drop="first"' prevents the dummy variable trap by dropping one of the encoded columns per category.
categorical_columns = ['batting_team', 'bowling_team', 'city']

transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, drop='first'), categorical_columns)
    ], 
    remainder='passthrough' # Leave the numerical columns (runs_left, crr, rrr, etc.) untouched
)

# 2. Build the Pipeline
# The data flows through the transformer first, then into the Logistic Regression classifier.
ml_pipeline = Pipeline(
    steps=[
        ('preprocessing', transformer),
        ('classifier', LogisticRegression(solver='liblinear')) # liblinear is efficient for this dataset size
    ]
)

# 3. Train the Model
print("Training the Logistic Regression model...")
ml_pipeline.fit(X_train, y_train)

# 4. Make Predictions on the Test Set
y_pred = ml_pipeline.predict(X_test)

# Extract the probability of winning (class 1)
y_pred_probabilities = ml_pipeline.predict_proba(X_test)[:, 1] 

# 5. Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_probabilities)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(f"Model ROC-AUC Score: {roc_auc:.4f}")

Training the Logistic Regression model...


ValueError: Input X contains infinity or a value too large for dtype('float64').

In [12]:
import numpy as np

print("--- Diagnostic Check ---")
print("1. Checking for Nulls in X_train:")
print(X_train.isnull().sum())

print("\n2. Checking for Infinite Values in X_train:")
# This checks the numerical columns for 'inf' which crashes ML models
num_cols = ['runs_left', 'balls_left', 'wickets_remaining', 'target', 'crr', 'rrr']
for col in num_cols:
    inf_count = np.isinf(X_train[col]).sum()
    print(f"{col} infinite values: {inf_count}")

print("\n3. Checking Data Types:")
print(X_train.dtypes)

--- Diagnostic Check ---
1. Checking for Nulls in X_train:
batting_team         0
bowling_team         0
city                 0
runs_left            0
balls_left           0
wickets_remaining    0
target               0
crr                  0
rrr                  0
dtype: int64

2. Checking for Infinite Values in X_train:
runs_left infinite values: 0
balls_left infinite values: 0
wickets_remaining infinite values: 0
target infinite values: 0
crr infinite values: 758
rrr infinite values: 0

3. Checking Data Types:
batting_team          object
bowling_team          object
city                  object
runs_left              int64
balls_left             int64
wickets_remaining      int64
target                 int64
crr                  float64
rrr                  float64
dtype: object


In [13]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Define the VIP list
final_columns = [
    'batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 
    'wickets_remaining', 'target', 'crr', 'rrr', 'result'
]

final_df = chase_df[final_columns].copy()

# 2. Handle missing cities
final_df['city'] = final_df['city'].fillna('Dubai')

# 3. THE FIX: Force any hidden Infinities to become Nulls, then drop all Nulls entirely
final_df.replace([np.inf, -np.inf], np.nan, inplace=True)
final_df.dropna(inplace=True)

# 4. Filter unrealistic scenarios
final_df = final_df[final_df['balls_left'] > 0]

# 5. Shuffle and Split
final_df = final_df.sample(final_df.shape[0], random_state=42)

X = final_df.drop(columns=['result'])
y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data is pure. Ready for the pipeline.")

Data is pure. Ready for the pipeline.


In [15]:
import pandas as pd # Ensure pandas is imported in this cell too

def predict_win_probability(batting_team, bowling_team, city, runs_left, balls_left, wickets, target):
    # Calculate crr and rrr for the input
    crr = ((target - runs_left) * 6) / (120 - balls_left)
    rrr = (runs_left * 6) / balls_left
    
    # Create a small dataframe for the model
    input_df = pd.DataFrame({
        'batting_team': [batting_team],
        'bowling_team': [bowling_team],
        'city': [city],
        'runs_left': [runs_left],
        'balls_left': [balls_left],
        'wickets_remaining': [wickets],
        'target': [target],
        'crr': [crr],
        'rrr': [rrr]
    })
    
    # Get probability from your trained ml_pipeline
    result = ml_pipeline.predict_proba(input_df)
    
    loss_prob = result[0][0]
    win_prob = result[0][1]
    
    print(f"{batting_team} Win Probability: {round(win_prob * 100, 2)}%")
    print(f"{bowling_team} Win Probability: {round(loss_prob * 100, 2)}%")

In [16]:
predict_win_probability('Mumbai Indians', 'Chennai Super Kings', 'Mumbai', 50, 30, 7, 180)

NotFittedError: Pipeline is not fitted yet.

In [17]:
# Make sure this specific cell runs without any red error messages!
ml_pipeline.fit(X_train, y_train) 
print("Model training complete!") # If you see this message, the error will go away

Model training complete!


In [18]:
# Run this cell again
predict_win_probability('Mumbai Indians', 'Chennai Super Kings', 'Mumbai', 50, 30, 7, 180)

Mumbai Indians Win Probability: 60.57%
Chennai Super Kings Win Probability: 39.43%


In [19]:
import pickle
import os

# Create the models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save the trained pipeline to a file
with open('models/ipl_win_predictor.pkl', 'wb') as file:
    pickle.dump(ml_pipeline, file)

print("Model successfully saved to models/ipl_win_predictor.pkl!")

Model successfully saved to models/ipl_win_predictor.pkl!


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Isolate the Pre-Match Data
# We drop any matches that ended in a 'no result' or washout (where winner is null)
pre_match_df = matches_df.dropna(subset=['winner', 'city']).copy()

# 2. Create the Target Variable
# If team1 won, the result is 1. If team2 won, the result is 0.
pre_match_df['result'] = pre_match_df.apply(lambda row: 1 if row['team1'] == row['winner'] else 0, axis=1)

# 3. Define Features (X) and Target (y)
features = ['team1', 'team2', 'city']
X_pre = pre_match_df[features]
y_pre = pre_match_df['result']

# 4. Train-Test Split
X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)

# 5. Build the Pre-Match Pipeline
# We use handle_unknown='ignore' so it doesn't crash if a completely new stadium is introduced
pre_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), features)
    ], 
    remainder='drop'
)

# We use a Random Forest here because it is excellent at capturing specific "matchup" interactions
# (e.g., Team A might be statistically worse overall, but has a specific historical advantage over Team B)
pre_match_pipeline = Pipeline(
    steps=[
        ('preprocessing', pre_transformer),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)) 
    ]
)

# 6. Train the Model
print("Training the Pre-Match Model...")
pre_match_pipeline.fit(X_train_pre, y_train_pre)

# 7. Evaluate the Model
y_pred_pre = pre_match_pipeline.predict(X_test_pre)
acc = accuracy_score(y_test_pre, y_pred_pre)

print(f"Pre-Match Model Accuracy: {acc * 100:.2f}%")

Training the Pre-Match Model...
Pre-Match Model Accuracy: 44.86%


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Isolate the Pre-Match Data
pre_match_df = matches_df.dropna(subset=['winner', 'city']).copy()

# 2. Create the Target Variable (1 if team1 won, 0 if team2 won)
pre_match_df['result'] = (pre_match_df['team1'] == pre_match_df['winner']).astype(int)

# 3. Define Features (X) and Target (y)
features = ['team1', 'team2', 'city']
X_pre = pre_match_df[features]
y_pre = pre_match_df['result']

X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)

# 4. Build the Revised Pipeline
pre_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, drop='first'), features)
    ], 
    remainder='drop'
)

# SWITCHED: Using Logistic Regression for better probability distribution
pre_match_pipeline = Pipeline(
    steps=[
        ('preprocessing', pre_transformer),
        ('classifier', LogisticRegression(solver='liblinear')) 
    ]
)

# 5. Train and Evaluate
print("Training the Revised Pre-Match Model...")
pre_match_pipeline.fit(X_train_pre, y_train_pre)

y_pred_pre = pre_match_pipeline.predict(X_test_pre)
acc = accuracy_score(y_test_pre, y_pred_pre)

print(f"Revised Pre-Match Model Accuracy: {acc * 100:.2f}%")

Training the Revised Pre-Match Model...
Revised Pre-Match Model Accuracy: 44.86%


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. Isolate the Pre-Match Data
base_df = matches_df.dropna(subset=['winner', 'city']).copy()

# 2. THE FIX: Create Data Mirroring
# Perspective 1: Normal
df1 = base_df[['team1', 'team2', 'city', 'winner']].copy()
df1['result'] = (df1['team1'] == df1['winner']).astype(int)

# Perspective 2: Swapped
df2 = base_df[['team2', 'team1', 'city', 'winner']].copy()
df2.columns = ['team1', 'team2', 'city', 'winner'] # Rename columns so they match df1
df2['result'] = (df2['team1'] == df2['winner']).astype(int)

# Combine both perspectives to create a perfectly balanced dataset
balanced_df = pd.concat([df1, df2]).sample(frac=1, random_state=42)

# 3. Define Features (X) and Target (y)
features = ['team1', 'team2', 'city']
X_pre = balanced_df[features]
y_pre = balanced_df['result']

X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)

# 4. Build the Pipeline
pre_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, drop='first'), features)
    ], 
    remainder='drop'
)

pre_match_pipeline = Pipeline(
    steps=[
        ('preprocessing', pre_transformer),
        ('classifier', LogisticRegression(solver='liblinear')) 
    ]
)

# 5. Train and Evaluate
print("Training the Mirrored Pre-Match Model...")
pre_match_pipeline.fit(X_train_pre, y_train_pre)

y_pred_pre = pre_match_pipeline.predict(X_test_pre)
acc = accuracy_score(y_test_pre, y_pred_pre)

print(f"Mirrored Model Accuracy: {acc * 100:.2f}%")

Training the Mirrored Pre-Match Model...
Mirrored Model Accuracy: 55.68%


In [23]:
import numpy as np
import pandas as pd
from itertools import permutations

# 1. Define Teams and their Home Cities
active_teams = [
    'Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans',
    'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians',
    'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru',
    'Sunrisers Hyderabad'
]

home_cities = {
    'Chennai Super Kings': 'Chennai', 'Delhi Capitals': 'Delhi', 'Gujarat Titans': 'Ahmedabad',
    'Kolkata Knight Riders': 'Kolkata', 'Lucknow Super Giants': 'Lucknow', 'Mumbai Indians': 'Mumbai',
    'Punjab Kings': 'Chandigarh', 'Rajasthan Royals': 'Jaipur', 'Royal Challengers Bengaluru': 'Bengaluru',
    'Sunrisers Hyderabad': 'Hyderabad'
}

# 2. Generate the Schedule and Pre-Calculate Probabilities
matchups = []
for home_team, away_team in permutations(active_teams, 2):
    matchups.append({'team1': home_team, 'team2': away_team, 'city': home_cities[home_team]})
    
schedule_df = pd.DataFrame(matchups)
# Predict probabilities for all 90 group stage matches at once for speed
schedule_df['team1_win_prob'] = pre_match_pipeline.predict_proba(schedule_df[['team1', 'team2', 'city']])[:, 1]

# Helper function for Playoff Matches (Assuming Ahmedabad hosts all playoffs)
def simulate_playoff_match(team1, team2):
    match_data = pd.DataFrame([{'team1': team1, 'team2': team2, 'city': 'Ahmedabad'}])
    prob = pre_match_pipeline.predict_proba(match_data)[0][1]
    return team1 if np.random.rand() < prob else team2

# 3. The Season Simulator
def simulate_season():
    points_table = {team: 0 for team in active_teams}
    
    # Simulate Group Stage
    for _, row in schedule_df.iterrows():
        if np.random.rand() < row['team1_win_prob']:
            points_table[row['team1']] += 2
        else:
            points_table[row['team2']] += 2
            
    # Sort points table to get Top 4
    standings = sorted(points_table.items(), key=lambda x: x[1], reverse=True)
    top_4 = [team for team, points in standings[:4]]
    
    # Simulate Playoffs
    q1_winner = simulate_playoff_match(top_4[0], top_4[1])
    q1_loser = top_4[1] if q1_winner == top_4[0] else top_4[0]
    
    elim_winner = simulate_playoff_match(top_4[2], top_4[3])
    q2_winner = simulate_playoff_match(q1_loser, elim_winner)
    
    champion = simulate_playoff_match(q1_winner, q2_winner)
    return champion

# 4. Run the Monte Carlo Loop 1000 Times
print("Running 1,000 IPL Season Simulations... (This will take a few seconds)")
championships_won = []

for _ in range(1000):
    championships_won.append(simulate_season())

# 5. Calculate and Display Final Percentages
results = pd.Series(championships_won).value_counts(normalize=True) * 100

print("\n🏆 --- IPL TOURNAMENT WIN PROBABILITIES --- 🏆")
print(results.round(2).astype(str) + '%')

Running 1,000 IPL Season Simulations... (This will take a few seconds)

🏆 --- IPL TOURNAMENT WIN PROBABILITIES --- 🏆
Gujarat Titans                 27.9%
Lucknow Super Giants           27.9%
Chennai Super Kings            18.8%
Mumbai Indians                  9.9%
Kolkata Knight Riders           4.9%
Rajasthan Royals                3.9%
Delhi Capitals                  1.9%
Royal Challengers Bengaluru     1.8%
Sunrisers Hyderabad             1.5%
Punjab Kings                    1.5%
Name: proportion, dtype: object


In [26]:
import requests
import zipfile
import io
import pandas as pd
import os

print("Initiating Extraction Phase...")

url = "https://cricsheet.org/downloads/ipl_male_csv2.zip"
response = requests.get(url)

all_2026_matches = []

# 1. EXTRACT: Unzip the file in memory and process individual match files
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # THE FIX: We added "and '_info' not in f" to skip the broken metadata files
    csv_files = [f for f in z.namelist() if f.endswith('.csv') and 'README' not in f and '_info' not in f]
    print(f"Found {len(csv_files)} historical match files. Filtering for 2026...")
    
    for file_name in csv_files:
        with z.open(file_name) as f:
            df = pd.read_csv(f)
            
            # Check if it's a 2026 match
            if 'start_date' in df.columns:
                df['start_date'] = pd.to_datetime(df['start_date'])
                df_2026 = df[df['start_date'].dt.year == 2026]
                
                # If it is 2026, add it to our list
                if not df_2026.empty:
                    all_2026_matches.append(df_2026)

if all_2026_matches:
    # Combine all 2026 matches into one master dataframe
    deliveries_2026 = pd.concat(all_2026_matches, ignore_index=True)
    print("Extraction complete. Initiating Transformation Phase...")
    
    # 2. TRANSFORM: Align the schema with Kaggle
    column_mapping = {
        'innings': 'inning',
        'striker': 'batter',
        'runs_off_bat': 'batsman_runs',
        'extras': 'extra_runs',
        'wicket_type': 'dismissal_kind'
    }
    deliveries_2026 = deliveries_2026.rename(columns=column_mapping)

    # Calculate 'total_runs' per ball
    deliveries_2026['total_runs'] = deliveries_2026['batsman_runs'] + deliveries_2026['extra_runs']

    # 3. LOAD: Save the clean 2026 data
    os.makedirs('data/raw_2026', exist_ok=True)
    deliveries_2026.to_csv('data/raw_2026/deliveries_2026.csv', index=False)

    print(f"Pipeline Successful! Extracted {len(deliveries_2026)} deliveries from the 2026 season.")
    display(deliveries_2026.head())
else:
    print("No 2026 matches found in the dataset. Cricsheet might still be uploading today's matches.")

Initiating Extraction Phase...
Found 1193 historical match files. Filtering for 2026...
Extraction complete. Initiating Transformation Phase...
Pipeline Successful! Extracted 5473 deliveries from the 2026 season.


,match_id,season,start_date,venue,inning,ball,batting_team,bowling_team,batter,non_striker,...,wides,noballs,byes,legbyes,penalty,dismissal_kind,player_dismissed,other_wicket_type,other_player_dismissed,total_runs
0,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.1,Sunrisers Hyderabad,Royal Challengers Bengaluru,TM Head,Abhishek Sharma,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.2,Sunrisers Hyderabad,Royal Challengers Bengaluru,TM Head,Abhishek Sharma,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.3,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.4,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
4,1527674,2026,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",1,0.5,Sunrisers Hyderabad,Royal Challengers Bengaluru,Abhishek Sharma,TM Head,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [27]:
import numpy as np

print("Initiating Feature Engineering for 2026...")

# 1. Standardize City Names (Cricsheet sometimes uses stadium names instead of cities)
venue_mapping = {
    'Wankhede Stadium, Mumbai': 'Mumbai', 'Wankhede Stadium': 'Mumbai',
    'M Chinnaswamy Stadium': 'Bengaluru', 'M.Chinnaswamy Stadium': 'Bengaluru',
    'Eden Gardens': 'Kolkata', 'Arun Jaitley Stadium': 'Delhi', 'Arun Jaitley Stadium, Delhi': 'Delhi',
    'MA Chidambaram Stadium': 'Chennai', 'MA Chidambaram Stadium, Chepauk, Chennai': 'Chennai',
    'Rajiv Gandhi International Stadium': 'Hyderabad', 'Narendra Modi Stadium, Ahmedabad': 'Ahmedabad',
    'Sawai Mansingh Stadium': 'Jaipur', 'Sawai Mansingh Stadium, Jaipur': 'Jaipur',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'Lucknow',
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur': 'Chandigarh',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'Dharamsala',
    'ACA-VDCA Cricket Stadium': 'Visakhapatnam'
}
deliveries_2026['city'] = deliveries_2026['venue'].map(venue_mapping).fillna(deliveries_2026['venue'])

# 2. Extract Match Targets & Winners directly from ball data
inning1 = deliveries_2026[deliveries_2026['inning'] == 1]
inning2 = deliveries_2026[deliveries_2026['inning'] == 2]

# Calculate Target (Inning 1 Score + 1)
targets = inning1.groupby('match_id')['total_runs'].sum().reset_index()
targets['target'] = targets['total_runs'] + 1

# Calculate Winners by comparing Inning 1 and Inning 2 scores
inning2_scores = inning2.groupby('match_id')['total_runs'].sum().reset_index()
match_scores = targets.merge(inning2_scores, on='match_id', how='left').fillna(0)

teams = deliveries_2026[['match_id', 'inning', 'batting_team', 'bowling_team']].drop_duplicates()
team1_info = teams[teams['inning'] == 1][['match_id', 'batting_team']].rename(columns={'batting_team': 'team1'})
team2_info = teams[teams['inning'] == 2][['match_id', 'batting_team']].rename(columns={'batting_team': 'team2'})

match_scores = match_scores.merge(team1_info, on='match_id').merge(team2_info, on='match_id', how='left')

def determine_winner(row):
    if row['total_runs_y'] >= row['target']: return row['team2']
    elif row['total_runs_x'] > row['total_runs_y']: return row['team1']
    else: return 'Tie'

match_scores['winner'] = match_scores.apply(determine_winner, axis=1)

# 3. Build the 2026 Run Chase DataFrame
chase_2026 = inning2.merge(match_scores[['match_id', 'target', 'winner']], on='match_id').copy()

chase_2026['current_score'] = chase_2026.groupby('match_id')['total_runs'].cumsum()
chase_2026['runs_left'] = chase_2026['target'] - chase_2026['current_score']
chase_2026['runs_left'] = chase_2026['runs_left'].apply(lambda x: x if x >= 0 else 0)

# Cricsheet over format: 0.1, 0.2, etc. (Convert to exact balls bowled)
chase_2026['over_num'] = chase_2026['ball'].astype(int)
chase_2026['ball_num'] = ((chase_2026['ball'] - chase_2026['over_num']) * 10).round().astype(int)
chase_2026['balls_bowled'] = (chase_2026['over_num'] * 6) + chase_2026['ball_num']
chase_2026['balls_left'] = 120 - chase_2026['balls_bowled']
chase_2026['balls_left'] = chase_2026['balls_left'].apply(lambda x: x if x >= 0 else 0)

chase_2026['is_wicket'] = chase_2026['player_dismissed'].notnull().astype(int)
chase_2026['wickets_remaining'] = 10 - chase_2026.groupby('match_id')['is_wicket'].cumsum()

# Mathematical Metrics
chase_2026['crr'] = (chase_2026['current_score'] * 6) / chase_2026['balls_bowled']
chase_2026['rrr'] = (chase_2026['runs_left'] * 6) / chase_2026['balls_left']
chase_2026['result'] = (chase_2026['batting_team'] == chase_2026['winner']).astype(int)

# 4. Isolate final columns and strictly clean math anomalies
final_cols = ['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 'wickets_remaining', 'target', 'crr', 'rrr', 'result']
clean_2026_df = chase_2026[final_cols].copy()

clean_2026_df.replace([np.inf, -np.inf], np.nan, inplace=True)
clean_2026_df.dropna(inplace=True)
clean_2026_df = clean_2026_df[clean_2026_df['balls_left'] > 0]

print(f"2026 Data Cleaned! Generated {len(clean_2026_df)} new run-chase scenarios.")
display(clean_2026_df.tail())

Initiating Feature Engineering for 2026...
2026 Data Cleaned! Generated 2596 new run-chase scenarios.


,batting_team,bowling_team,city,runs_left,balls_left,wickets_remaining,target,crr,rrr,result
2602,Punjab Kings,Mumbai Indians,Mumbai,7,23,7,196,11.690722,1.826087,1
2603,Punjab Kings,Mumbai Indians,Mumbai,7,22,7,196,11.571429,1.909091,1
2604,Punjab Kings,Mumbai Indians,Mumbai,3,21,7,196,11.696970,0.857143,1
2605,Punjab Kings,Mumbai Indians,Mumbai,2,20,7,196,11.640000,0.600000,1
2606,Punjab Kings,Mumbai Indians,Mumbai,0,19,7,196,11.762376,0.000000,1


In [29]:
print("Extracting 2026 Match Metadata...")

# 1. Get the city for each match from our ball-by-ball data
match_cities = deliveries_2026.groupby('match_id')['city'].first().reset_index()

# 2. Combine it with the team and winner data we calculated in the previous cell
matches_2026_df = match_scores[['match_id', 'team1', 'team2', 'winner']].copy()
matches_2026_df = matches_2026_df.merge(match_cities, on='match_id')

# 3. Reorder columns to match your historical data
matches_2026_df = matches_2026_df[['match_id', 'city', 'team1', 'team2', 'winner']]

# 4. Drop any 'Tie' or unfinished matches to keep the machine learning math clean
matches_2026_df = matches_2026_df[matches_2026_df['winner'] != 'Tie']

# 5. Save it to your local directory just in case you need it later
matches_2026_df.to_csv('data/raw_2026/matches_2026.csv', index=False)

print(f"Successfully generated metadata for {len(matches_2026_df)} matches from the 2026 season!")
display(matches_2026_df.head())
display(matches_2026_df.tail())

Extracting 2026 Match Metadata...
Successfully generated metadata for 24 matches from the 2026 season!


,match_id,city,team1,team2,winner
0,1527674,"M Chinnaswamy Stadium, Bengaluru",Sunrisers Hyderabad,Royal Challengers Bengaluru,Royal Challengers Bengaluru
1,1527675,Mumbai,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians
2,1527676,"Barsapara Cricket Stadium, Guwahati",Chennai Super Kings,Rajasthan Royals,Rajasthan Royals
3,1527677,Maharaja Yadavindra Singh International Cricke...,Gujarat Titans,Punjab Kings,Punjab Kings
4,1527678,Lucknow,Lucknow Super Giants,Delhi Capitals,Delhi Capitals


,match_id,city,team1,team2,winner
19,1527693,Mumbai,Royal Challengers Bengaluru,Mumbai Indians,Royal Challengers Bengaluru
20,1529264,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Sunrisers Hyderabad,Rajasthan Royals,Sunrisers Hyderabad
21,1529265,Chennai,Chennai Super Kings,Kolkata Knight Riders,Chennai Super Kings
22,1529266,"M Chinnaswamy Stadium, Bengaluru",Lucknow Super Giants,Royal Challengers Bengaluru,Royal Challengers Bengaluru
23,1529267,Mumbai,Mumbai Indians,Punjab Kings,Punjab Kings


In [30]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("--- PHASE 1: RETRAINING THE IN-PLAY RUN CHASE MODEL ---")

# 1. Merge the historical run chase data with the new 2026 data
combined_chase_df = pd.concat([final_df, clean_2026_df], ignore_index=True)
combined_chase_df = combined_chase_df.sample(frac=1, random_state=42) # Shuffle

# 2. Split and Train
X_chase = combined_chase_df.drop(columns=['result'])
y_chase = combined_chase_df['result']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_chase, y_chase, test_size=0.2, random_state=42)

ml_pipeline.fit(X_train_c, y_train_c)
acc_c = accuracy_score(y_test_c, ml_pipeline.predict(X_test_c))
print(f"Upgraded In-Play Model Accuracy: {acc_c * 100:.2f}%")


print("\n--- PHASE 2: RETRAINING THE PRE-MATCH TOURNAMENT MODEL ---")

# 1. Merge historical matches with 2026 matches
combined_matches_df = pd.concat([base_df[['team1', 'team2', 'city', 'winner']], matches_2026_df], ignore_index=True)

# 2. Data Mirroring (to prevent column bias, as we did before)
df1 = combined_matches_df.copy()
df1['result'] = (df1['team1'] == df1['winner']).astype(int)

df2 = combined_matches_df[['team2', 'team1', 'city', 'winner']].copy()
df2.columns = ['team1', 'team2', 'city', 'winner']
df2['result'] = (df2['team1'] == df2['winner']).astype(int)

balanced_matches = pd.concat([df1, df2]).sample(frac=1, random_state=42)

# 3. Split and Train
X_pre = balanced_matches[['team1', 'team2', 'city']]
y_pre = balanced_matches['result']
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)

pre_match_pipeline.fit(X_train_p, y_train_p)
acc_p = accuracy_score(y_test_p, pre_match_pipeline.predict(X_test_p))
print(f"Upgraded Pre-Match Model Accuracy: {acc_p * 100:.2f}%")


print("\n--- PHASE 3: EXPORTING UPGRADED MODELS ---")

# Save the updated Run Chase pipeline
with open('models/ipl_win_predictor.pkl', 'wb') as file:
    pickle.dump(ml_pipeline, file)

# Save the updated Tournament pipeline
with open('models/ipl_tournament_predictor.pkl', 'wb') as file:
    pickle.dump(pre_match_pipeline, file)

print("SUCCESS! Both 2026-aware models are locked, loaded, and saved to the 'models' folder.")

--- PHASE 1: RETRAINING THE IN-PLAY RUN CHASE MODEL ---
Upgraded In-Play Model Accuracy: 80.75%

--- PHASE 2: RETRAINING THE PRE-MATCH TOURNAMENT MODEL ---


ValueError: Found unknown categories [nan] in column 1 during transform

In [31]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("--- PHASE 1: RETRAINING THE IN-PLAY RUN CHASE MODEL ---")

# 1. Merge and shuffle the run chase data
combined_chase_df = pd.concat([final_df, clean_2026_df], ignore_index=True)
combined_chase_df = combined_chase_df.dropna() # Ensure absolute purity
combined_chase_df = combined_chase_df.sample(frac=1, random_state=42) 

# 2. Split and Train
X_chase = combined_chase_df.drop(columns=['result'])
y_chase = combined_chase_df['result']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_chase, y_chase, test_size=0.2, random_state=42)

ml_pipeline.fit(X_train_c, y_train_c)
acc_c = accuracy_score(y_test_c, ml_pipeline.predict(X_test_c))
print(f"Upgraded In-Play Model Accuracy: {acc_c * 100:.2f}%")


print("\n--- PHASE 2: RETRAINING THE PRE-MATCH TOURNAMENT MODEL ---")

# 1. Merge historical matches with 2026 matches
combined_matches_df = pd.concat([base_df[['team1', 'team2', 'city', 'winner']], matches_2026_df], ignore_index=True)

# THE FIX: Drop any rows where a team or city is NaN (e.g., rain washouts)
combined_matches_df = combined_matches_df.dropna() 

# 2. Data Mirroring (to prevent column bias)
df1 = combined_matches_df.copy()
df1['result'] = (df1['team1'] == df1['winner']).astype(int)

df2 = combined_matches_df[['team2', 'team1', 'city', 'winner']].copy()
df2.columns = ['team1', 'team2', 'city', 'winner']
df2['result'] = (df2['team1'] == df2['winner']).astype(int)

balanced_matches = pd.concat([df1, df2]).sample(frac=1, random_state=42)

# 3. Split and Train
X_pre = balanced_matches[['team1', 'team2', 'city']]
y_pre = balanced_matches['result']
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pre, y_pre, test_size=0.2, random_state=42)

pre_match_pipeline.fit(X_train_p, y_train_p)
acc_p = accuracy_score(y_test_p, pre_match_pipeline.predict(X_test_p))
print(f"Upgraded Pre-Match Model Accuracy: {acc_p * 100:.2f}%")


print("\n--- PHASE 3: EXPORTING UPGRADED MODELS ---")

# Save the updated Run Chase pipeline
with open('models/ipl_win_predictor.pkl', 'wb') as file:
    pickle.dump(ml_pipeline, file)

# Save the updated Tournament pipeline
with open('models/ipl_tournament_predictor.pkl', 'wb') as file:
    pickle.dump(pre_match_pipeline, file)

print("SUCCESS! Both 2026-aware models are locked, loaded, and saved to the 'models' folder.")

--- PHASE 1: RETRAINING THE IN-PLAY RUN CHASE MODEL ---
Upgraded In-Play Model Accuracy: 80.75%

--- PHASE 2: RETRAINING THE PRE-MATCH TOURNAMENT MODEL ---
Upgraded Pre-Match Model Accuracy: 60.00%

--- PHASE 3: EXPORTING UPGRADED MODELS ---
SUCCESS! Both 2026-aware models are locked, loaded, and saved to the 'models' folder.


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pickle

print("--- PATCHING THE TOURNAMENT PREDICTOR ---")

# 1. THE FIX: Change to handle_unknown='ignore'
pre_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['team1', 'team2', 'city'])
    ], 
    remainder='drop'
)

# 2. Rebuild Pipeline
pre_match_pipeline = Pipeline(
    steps=[
        ('preprocessing', pre_transformer),
        ('classifier', LogisticRegression(solver='liblinear')) 
    ]
)

# 3. Retrain on the balanced data we already created earlier
X_pre = balanced_matches[['team1', 'team2', 'city']]
y_pre = balanced_matches['result']
pre_match_pipeline.fit(X_pre, y_pre)

# 4. Overwrite the saved model
with open('models/ipl_tournament_predictor.pkl', 'wb') as file:
    pickle.dump(pre_match_pipeline, file)

print("✅ Model patched and saved! It will no longer crash on unknown cities.")

--- PATCHING THE TOURNAMENT PREDICTOR ---
✅ Model patched and saved! It will no longer crash on unknown cities.
